In [0]:
from pyspark.sql import functions as F

orders = spark.table("workspace.default.silver_orders")
items = spark.table("workspace.default.silver_order_items")

gold_monthly_sales = (
    orders
    .filter(F.col("order_status") == "delivered")
    .join(items, on="order_id", how="inner")
    .withColumn(
        "order_month",
        F.date_trunc("month", F.col("order_purchase_timestamp"))
    )
    .groupBy("order_month")
    .agg(
        F.sum("price").alias("total_sales"),
        F.sum("freight_value").alias("total_freight"),
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("total_items")
    )
    .orderBy("order_month")
)

(
    gold_monthly_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_monthly_sales")
)

display(gold_monthly_sales)

order_month,total_sales,total_freight,total_orders,total_items
2016-09-01T00:00:00.000Z,134.97,8.49,1,3
2016-10-01T00:00:00.000Z,40325.10999999997,6165.549999999999,265,313
2016-12-01T00:00:00.000Z,10.9,8.72,1,1
2017-01-01T00:00:00.000Z,111798.36000000022,15684.009999999998,750,913
2017-02-01T00:00:00.000Z,234223.4000000005,37015.92000000001,1653,1858
2017-03-01T00:00:00.000Z,359198.84999999974,55132.09999999999,2546,2897
2017-04-01T00:00:00.000Z,340669.68000000005,50142.72,2303,2569
2017-05-01T00:00:00.000Z,489338.249999999,77513.15000000005,3546,4004
2017-06-01T00:00:00.000Z,421923.3699999995,68127.00000000013,3135,3489
2017-07-01T00:00:00.000Z,481604.5199999985,84694.56000000006,3872,4416


In [0]:
from pyspark.sql import functions as F

items = spark.table("workspace.default.silver_order_items")
products = spark.table("workspace.default.silver_products")
translations = spark.table("workspace.default.silver_category_translation")

# Vendas por categoria de produto
gold_sales_by_category = (
    items
    .join(products, on="product_id", how="left")
    .join(translations, on="product_category_name", how="left")
    .groupBy(
        F.coalesce(
            F.col("product_category_name_english"),
            F.col("product_category_name"),
            F.lit("uncategorized")
        ).alias("product_category")
    )
    .agg(
        F.sum("price").alias("total_sales"),
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("total_items")
    )
    .orderBy(F.desc("total_sales"))
)

(
    gold_sales_by_category.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_sales_by_category")
)

# Vendas por vendedor
gold_sales_by_seller = (
    items
    .groupBy("seller_id")
    .agg(
        F.sum("price").alias("total_sales"),
        F.sum("freight_value").alias("total_freight"),
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("total_items")
    )
    .orderBy(F.desc("total_sales"))
)

(
    gold_sales_by_seller.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_sales_by_seller")
)

print("Tabelas Gold criadas com sucesso!")
display(gold_sales_by_category.limit(10))
display(gold_sales_by_seller.limit(10))

Tabelas Gold criadas com sucesso!


product_category,total_sales,total_orders,total_items
health_beauty,1258681.3399999922,8836,9670
watches_gifts,1205005.6799999962,5624,5991
bed_bath_table,1036988.6799999786,9417,11115
sports_leisure,988048.969999985,7720,8641
computers_accessories,911954.3199999894,6689,7827
furniture_decor,729762.4899999906,6449,8334
cool_stuff,635290.849999998,3632,3796
housewares,632248.6599999944,5884,6964
auto,592720.1099999971,3897,4235
garden_tools,485256.4599999986,3518,4347


seller_id,total_sales,total_freight,total_orders,total_items
4869f7a5dfa277a7dca6462dcf3b52b2,229472.62999999808,20168.070000000003,1132,1156
53243585a1d6dc2643021fd1853d8905,222776.04999999967,13080.630000000003,358,410
4a3ca9315b744ce9f8e9374361493884,200472.91999999547,35067.04000000008,1806,1987
fa1c13f2614d7b5c4749cbc52fecda94,194042.02999999849,10042.699999999992,585,586
7c67e1448b00f6e969d365cea6b010ab,187923.88999999937,51612.54999999991,982,1364
7e93a43ef30c4f03f38b393420bc753a,176431.86999999994,6322.1799999999985,336,340
da8622b14eb17ae2831f4ac5b9dab84a,160236.5699999963,24955.75000000006,1314,1551
7a67c85e85bb2ce8582c35f2203ad736,141745.53000000166,20902.85000000001,1160,1171
1025f0e2d44d7041d6cf58b6550e0bfa,138968.55000000002,33892.14000000002,915,1428
955fee9216a65b617aa5c0531780ce60,135171.6999999995,25430.980000000098,1287,1499


In [0]:
from pyspark.sql import functions as F

# 1. Verificar chaves nulas e duplicadas nas tabelas principais
checks = {
    "silver_orders": ["order_id"],
    "silver_customers": ["customer_id"],
    "silver_order_items": ["order_id", "order_item_id"],
    "silver_products": ["product_id"],
    "silver_sellers": ["seller_id"],
}

for table_name, keys in checks.items():
    df = spark.table(f"workspace.default.{table_name}")

    null_condition = F.lit(False)
    for key in keys:
        null_condition = null_condition | F.col(key).isNull()

    null_keys = df.filter(null_condition).count()
    duplicate_keys = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(
        f"{table_name}: "
        f"chaves nulas = {null_keys}, "
        f"chaves duplicadas = {duplicate_keys}"
    )

# 2. Verificar valores financeiros inválidos
items = spark.table("workspace.default.silver_order_items")

invalid_values = items.filter(
    F.col("price").isNull()
    | (F.col("price") < 0)
    | F.col("freight_value").isNull()
    | (F.col("freight_value") < 0)
).count()

print("Itens com valores financeiros inválidos:", invalid_values)

# 3. Verificar consistência básica das tabelas Gold
for table_name in [
    "gold_monthly_sales",
    "gold_sales_by_category",
    "gold_sales_by_seller",
]:
    df = spark.table(f"workspace.default.{table_name}")
    print(f"{table_name}: {df.count()} registros")

silver_orders: chaves nulas = 0, chaves duplicadas = 0
silver_customers: chaves nulas = 0, chaves duplicadas = 0
silver_order_items: chaves nulas = 0, chaves duplicadas = 0
silver_products: chaves nulas = 0, chaves duplicadas = 0
silver_sellers: chaves nulas = 0, chaves duplicadas = 0
Itens com valores financeiros inválidos: 0
gold_monthly_sales: 23 registros
gold_sales_by_category: 74 registros
gold_sales_by_seller: 3095 registros
